# Ciência de Dados 1 — Aula 05: Validação hold-out e Pipeline (laboratório)

Cada tópico traz um **exemplo pronto** e um **exercício** logo abaixo. Rode as células em ordem.

**Arquivo usado:** `pedidos_aula05.csv` (pedidos com data, atributos e o alvo `atrasado`).

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1) Ler e inspecionar

**CSV usado:** `pedidos_aula05.csv`  
Pedidos com data, valor, itens, distância, canal, região e o alvo `atrasado` (0/1).

**Exemplo:**

In [ ]:
ped = pd.read_csv("pedidos_aula05.csv", parse_dates=["data"])
print(ped.shape)
ped.head()

**Exercício 1:** Mostre a proporção de pedidos atrasados (média de `atrasado`).

In [ ]:
# TODO — resolva aqui


## 2) Separar X e y

**CSV usado:** `pedidos_aula05.csv`  
Guardamos as colunas numéricas e a categórica separadas (a categórica NÃO é número).

**Exemplo:**

In [ ]:
num = ["valor","itens","distancia_km"]
cat = ["canal","regiao"]
X = ped[num + cat]
y = ped["atrasado"]
Xn = ped[num]   # só numéricas, para os primeiros passos
print(X.shape, y.shape)

**Exercício 2:** Mostre quantos valores diferentes existem em cada coluna categórica (canal e regiao).

In [ ]:
# TODO — resolva aqui


## 3) Hold-out: treino e teste

**CSV usado:** `pedidos_aula05.csv`  
Reservamos parte dos dados para estimar a generalização.

**Exemplo:**

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xn, y, test_size=0.2, random_state=0, stratify=y)
print("treino:", Xtr.shape[0], "teste:", Xte.shape[0])

**Exercício 3:** Refaça a divisão com test_size=0.3 e mostre os tamanhos.

In [ ]:
# TODO — resolva aqui


## 4) Três conjuntos: treino / validação / teste

**CSV usado:** `pedidos_aula05.csv`  
Dois cortes encadeados: primeiro separa o teste; depois divide o resto em treino e validação.

**Exemplo:**

In [ ]:
Xrest, Xte, yrest, yte = train_test_split(Xn, y, test_size=0.2, random_state=0, stratify=y)
Xtr, Xval, ytr, yval = train_test_split(Xrest, yrest, test_size=0.25, random_state=0, stratify=yrest)
print("treino:", len(Xtr), "val:", len(Xval), "teste:", len(Xte))

**Exercício 4:** Confirme que treino + validação + teste somam o total de linhas.

In [ ]:
# TODO — resolva aqui


## 5) A semente fixa o corte

**CSV usado:** `pedidos_aula05.csv`  
Mesmo random_state = mesma divisão (reprodutível).

**Exemplo:**

In [ ]:
a = train_test_split(Xn, y, test_size=0.2, random_state=42)[0].index
b = train_test_split(Xn, y, test_size=0.2, random_state=42)[0].index
print("mesmos índices?", (a == b).all())

**Exercício 5:** Mostre que com random_state diferente (1 e 2) os índices de treino MUDAM.

In [ ]:
# TODO — resolva aqui


## 6) Um corte só é arriscado

**CSV usado:** `pedidos_aula05.csv`  
A acurácia muda conforme a sorte do corte — a estimativa tem variância.

**Exemplo:**

In [ ]:
def acc(seed):
    a,b,c,d = train_test_split(Xn, y, test_size=0.2, random_state=seed, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(a, c)
    return accuracy_score(d, m.predict(b))
print([round(acc(s),3) for s in range(5)])

**Exercício 6:** Calcule a MÉDIA e o desvio-padrão das acurácias das 5 sementes acima.

In [ ]:
# TODO — resolva aqui


## 7) Vazamento: padronizar na hora errada

**CSV usado:** `pedidos_aula05.csv`  
Padronizar usando TODO o dado (antes de separar) deixa o teste 'espiar' o treino.

**Exemplo:**

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xn, y, test_size=0.2, random_state=0, stratify=y)
sc = StandardScaler().fit(Xtr)          # certo: ajusta SÓ no treino
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
print("média do teste padronizado (com stats do treino):", Xte_s.mean(axis=0).round(2))

**Exercício 7:** Explique num comentário por que ajustar o StandardScaler em Xn INTEIRO (treino+teste) é vazamento.

In [ ]:
# TODO — resolva aqui


## 8) Codificar a categórica (OneHot)

**CSV usado:** `pedidos_aula05.csv`  
Modelos não leem texto: transformamos canal/região em colunas 0/1.

**Exemplo:**

In [ ]:
oh = OneHotEncoder(handle_unknown="ignore")
ex = oh.fit_transform(ped[["canal"]]).toarray()[:3]
print(oh.get_feature_names_out(["canal"]))
print(ex)

**Exercício 8:** Aplique o OneHotEncoder na coluna `regiao` e mostre os nomes das colunas geradas.

In [ ]:
# TODO — resolva aqui


## 9) ColumnTransformer: cada tipo de coluna

**CSV usado:** `pedidos_aula05.csv`  
Padroniza as numéricas e faz OneHot nas categóricas, tudo junto.

**Exemplo:**

In [ ]:
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print("colunas após transformar:", pre.fit_transform(Xtr).shape[1])

**Exercício 9:** Quantas colunas o ColumnTransformer gera? (3 numéricas + as dummies de canal e regiao)

In [ ]:
# TODO — resolva aqui


## 10) Pipeline: pré-processamento + modelo

**CSV usado:** `pedidos_aula05.csv`  
Um objeto só encadeia o pré-processamento e o classificador.

**Exemplo:**

In [ ]:
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
pipe.fit(Xtr, ytr)
print("acurácia no teste:", round(accuracy_score(yte, pipe.predict(Xte)), 3))

**Exercício 10:** Troque o modelo por uma árvore (DecisionTreeClassifier, random_state=0) no Pipeline e avalie no teste.

In [ ]:
# TODO — resolva aqui


## 11) O Pipeline evita vazamento na validação cruzada

**CSV usado:** `pedidos_aula05.csv`  
No cross-validation, o pré-processamento é refeito DENTRO de cada dobra — sem vazar.

**Exemplo:**

In [ ]:
scores = cross_val_score(pipe, Xtr, ytr, cv=5, scoring="accuracy")
print("acurácias por dobra:", scores.round(3))
print("média:", round(scores.mean(), 3))

**Exercício 11:** Rode a validação cruzada (cv=5) do pipeline da árvore e mostre a média.

In [ ]:
# TODO — resolva aqui


## 12) Reprodutibilidade: mesma receita, mesmo resultado

**CSV usado:** `pedidos_aula05.csv`  
Fixando as sementes, o pipeline treina igual toda vez.

**Exemplo:**

In [ ]:
def treina():
    p = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
    return p.fit(Xtr, ytr).predict(Xte)
print("previsões idênticas nas duas execuções?", (treina() == treina()).all())

**Exercício 12:** Mostre que a acurácia é exatamente a mesma nas duas execuções (use accuracy_score).

In [ ]:
# TODO — resolva aqui


## 13) Imputar ausentes DENTRO do pipeline

**CSV usado:** `pedidos_aula05.csv`  
Se faltar valor, o pipeline imputa usando só o treino — sem vazamento.

**Exemplo:**

In [ ]:
Xmiss = X.copy()
Xmiss.loc[Xmiss.sample(30, random_state=0).index, "distancia_km"] = np.nan
pre_imp = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
Xtr2, Xte2, ytr2, yte2 = train_test_split(Xmiss, y, test_size=0.2, random_state=0, stratify=y)
pipe_imp = Pipeline([("pre", pre_imp), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr2, ytr2)
print("rodou com ausentes; acurácia:", round(accuracy_score(yte2, pipe_imp.predict(Xte2)), 3))

**Exercício 13:** Confirme que Xmiss tem valores ausentes em distancia_km (conte os NaN).

In [ ]:
# TODO — resolva aqui


## 14) Cada decisão gasta um conjunto

**CSV usado:** `pedidos_aula05.csv`  
Escolhemos o modelo olhando a VALIDAÇÃO; o teste fica intocado até o fim.

**Exemplo:**

In [ ]:
Xrest, Xte, yrest, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
Xtr, Xval, ytr, yval = train_test_split(Xrest, yrest, test_size=0.25, random_state=0, stratify=yrest)
p1 = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr, ytr)
from sklearn.tree import DecisionTreeClassifier
p2 = Pipeline([("pre", pre), ("clf", DecisionTreeClassifier(max_depth=4, random_state=0))]).fit(Xtr, ytr)
print("val LR:", round(accuracy_score(yval, p1.predict(Xval)),3), "| val árvore:", round(accuracy_score(yval, p2.predict(Xval)),3))

**Exercício 14:** Escolha o melhor pela VALIDAÇÃO e só então meça esse vencedor no TESTE (uma vez).

In [ ]:
# TODO — resolva aqui


## 15) Hold-out no tempo: treinar no passado

**CSV usado:** `pedidos_aula05.csv`  
Com data, o teste deve ser o FUTURO. Ordenamos por data e cortamos por tempo.

**Exemplo:**

In [ ]:
ped_t = ped.sort_values("data").reset_index(drop=True)
corte = int(len(ped_t) * 0.8)
treino_t = ped_t.iloc[:corte]
teste_t  = ped_t.iloc[corte:]
print("treino até", treino_t["data"].max().date(), "| teste desde", teste_t["data"].min().date())

**Exercício 15:** Mostre a proporção de atraso no treino e no teste temporais (elas diferem — a distribuição mudou no tempo).

In [ ]:
# TODO — resolva aqui


## 16) Split temporal × aleatório

**CSV usado:** `pedidos_aula05.csv`  
O split aleatório mistura datas e fica otimista; o temporal imita o uso real.

**Exemplo:**

In [ ]:
Xtr_t, ytr_t = treino_t[num+cat], treino_t["atrasado"]
Xte_t, yte_t = teste_t[num+cat], teste_t["atrasado"]
pipe_t = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr_t, ytr_t)
acc_temp = accuracy_score(yte_t, pipe_t.predict(Xte_t))
print("acurácia TEMPORAL (treina passado, testa futuro):", round(acc_temp, 3))

**Exercício 16:** Compare com um split ALEATÓRIO (random_state=0) usando o mesmo pipeline. Qual fica mais otimista?

In [ ]:
# TODO — resolva aqui


## 17) Drift: os dados mudam no tempo

**CSV usado:** `pedidos_aula05.csv`  
Comparar treino (passado) com dados recentes revela deriva na distribuição.

**Exemplo:**

In [ ]:
print("valor médio — treino:", round(treino_t["valor"].mean(),1), "| recente:", round(teste_t["valor"].mean(),1))
print("atraso médio — treino:", round(treino_t["atrasado"].mean(),3), "| recente:", round(teste_t["atrasado"].mean(),3))

**Exercício 17:** Compare a média de `itens` entre treino e recente e diga se houve mudança perceptível.

In [ ]:
# TODO — resolva aqui


## 18) Estratificar mantém a proporção

**CSV usado:** `pedidos_aula05.csv`  
stratify=y garante a mesma taxa de atraso no treino e no teste.

**Exemplo:**

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print("treino:", round(ytr.mean(),3), "| teste:", round(yte.mean(),3))

**Exercício 18:** Faça a divisão SEM stratify (random_state=7) e compare as proporções — elas batem tão bem?

In [ ]:
# TODO — resolva aqui


## 19) Prever um pedido novo

**CSV usado:** `pedidos_aula05.csv`  
Com o pipeline treinado, um caso novo passa pelo MESMO pré-processamento.

**Exemplo:**

In [ ]:
novo = pd.DataFrame([{"valor":150,"itens":6,"distancia_km":12.0,"canal":"app","regiao":"N"}])
print("prob. de atraso:", round(pipe.predict_proba(novo)[0,1], 3))

**Exercício 19:** Preveja a probabilidade de atraso de um pedido: valor=90, itens=2, distancia=4, canal='loja', regiao='S'.

In [ ]:
# TODO — resolva aqui


## 20) Montando tudo: o fluxo correto

**CSV usado:** `pedidos_aula05.csv`  
Teste separado primeiro; pipeline ajustado só no treino; teste tocado uma vez.

**Exemplo:**

In [ ]:
Xrest, Xte, yrest, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
final = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xrest, yrest)
print("acurácia final (teste intocado):", round(accuracy_score(yte, final.predict(Xte)), 3))

**Exercício 20:** Reajuste o pipeline final em TODOS os dados (X, y) — é o modelo que iria para produção.

In [ ]:
# TODO — resolva aqui
